In [ ]:
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import snapatac2 as snap
import numpy as np
import pandas as pd
import os
import scanpy.external as sce
adata_concat = snap.read_dataset('output/mouse_brain.h5ads')

In [ ]:
data = adata_concat

In [ ]:
snap.tl.umap(data, use_rep="X_spectral_mnn")
snap.pl.umap(data, color="sample", interactive=False)


In [ ]:
snap.tl.umap(data, use_rep="X_spectral_harmony")
snap.pl.umap(data, color="sample", interactive=False)


In [ ]:
snap.pl.umap(data, color="sample")


In [ ]:
all_obs_names = data.obs_names

In [ ]:
df_PFC_annotation = pd.read_csv("output/merged-all-pfc-annotated.csv", index_col=0)

In [ ]:
df_WTatacPFC = df_PFC_annotation[df_PFC_annotation['batch'] == 'atac']

In [ ]:
df_WTatacPFC.dropna(axis=1,how='all',inplace=True)

In [ ]:
df_WTatacPFC.index= df_WTatacPFC.index.str.replace('WT_','')

In [ ]:
df_WTatacPFC.index = df_WTatacPFC.index.str.replace('-3','')

In [ ]:
df_WTatacPFC.to_csv("/data2st1/junyi/atac_annotation/pre_annotation.csv")

In [ ]:
sample_name = [item.split("/")[6]+item.split(".gz")[-1] for item in all_obs_names]

In [ ]:
data.obs['sample_barcode'] = sample_name

In [ ]:
data.obs['obs_names'] = data.obs_names

In [ ]:
data.obs_names = data.obs['sample_barcode']

In [ ]:
ct_pfc_p = df_WTatacPFC[["celltype.L2.p","celltype.L1.p"]]

In [ ]:
ct_pfc_p.index = "WT_W26_0_2_Perfrontal:"+ct_pfc_p.index

In [ ]:
ct_pfc_p

In [ ]:
df_obsnames = pd.DataFrame(data.obs_names)

In [ ]:
df_obsnames.columns = ['obs_names']
df_obsnames.set_index('obs_names',inplace=True)

In [ ]:
df_merge_label = df_obsnames.merge(ct_pfc_p,left_index=True,right_index=True,how='left')

In [ ]:
df_merge_label['celltype.L2.p'].fillna("Unknown").values.tolist()

In [ ]:
data.obs['celltype.L2.p'] = df_merge_label['celltype.L2.p'].fillna("Unknown").values.tolist()
data.obs['celltype.L1.p'] = df_merge_label['celltype.L1.p'].fillna("Unknown").values.tolist()

In [ ]:
snap.pl.umap(data, color="celltype.L1.p",marker_size=3)


In [ ]:
snap.pl.umap(data, color="celltype.L2.p",marker_size=3)


In [ ]:
snap.pl.umap(data, color="sample",marker_size=3)


In [ ]:
%%time
snap.pp.knn(data)
snap.tl.leiden(data)


In [ ]:
snap.pl.umap(data, color='leiden', marker_size=3, height=500)


In [ ]:
ct_pfc_p.index

In [ ]:
adata_concat.obs_names

In [ ]:
df_merge_inner = df_obsnames.merge(ct_pfc_p,left_index=True,right_index=True,how='inner')

In [ ]:
df_merge_label

In [ ]:
data.obs_names[-1]

In [ ]:
sum(data.obs_names == df_merge_label.index) == data.shape[0]

In [ ]:
selected_rows = df_merge_label.reset_index().loc[df_merge_label.reset_index().obs_names.str.contains("WT_W26_0_2_Perfrontal")].index

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
y_train = df_merge_label.iloc[selected_rows]["celltype.L1.p"].values
le = LabelEncoder()
y_train = le.fit_transform(y_train)
X_train = adata_concat.obsm['X_umap'][selected_rows]
#X_train = adata_concat.obsm['X_spectral_harmony'][selected_rows,:5]

#X_train = np.concatenate([X_train,adata_concat.obsm['X_spectral_harmony'][selected_rows,:5]],axis=1)

In [ ]:
# clf = SVC(C=1)
# clf.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
# Assuming X contains the features and y contains the target variable

# Initialize your model, for example, a Random Forest classifier
model =  GradientBoostingClassifier()

# Perform 5-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='accuracy')

# Print the cross-validation scores
print("Cross-validation scores:", scores)
print("Mean accuracy: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))



In [ ]:
clf = GradientBoostingClassifier()
clf.fit(X_train, y_train)

In [ ]:
y_valid = clf.predict(X_train)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

import matplotlib.pyplot as plt

# Generate confusion matrix
cm = confusion_matrix(y_train, y_valid)

# Plot confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix between y_valid and y_train')
plt.show()

In [ ]:
import copy
clss = copy.deepcopy(le.classes_)

In [ ]:
clss[-1]='UNknown'

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_train, y_valid, target_names=clss))

In [ ]:
pred_rows = df_merge_label.reset_index().loc[~df_merge_label.reset_index().obs_names.str.contains("WT_W26_0_2_Perfrontal")].index

In [ ]:
pred_rows

In [ ]:
#X_test = adata_concat.obsm['X_spectral_harmony'][pred_rows,:5]

X_test = adata_concat.obsm['X_umap'][pred_rows]
#X_test = np.concatenate([X_test,adata_concat.obsm['X_spectral_harmony'][pred_rows,:5]],axis=1)
y_test = clf.predict(X_test)

In [ ]:
y_test = le.inverse_transform(y_test)

In [ ]:
y_test

In [ ]:
set(y_test)

In [ ]:
all_rows = np.concatenate([pred_rows,selected_rows])

In [ ]:
le.inverse_transform(y_train)

In [ ]:
y_test.shape

In [ ]:
all_labels = le.inverse_transform(y_train).tolist() + y_test.tolist()

In [ ]:
df_all_labels = pd.DataFrame({'rows':all_rows,'labels':all_labels})

In [ ]:
df_all_labels.set_index('rows',inplace=True)    

In [ ]:
df_all_labels.sort_index(inplace=True)

In [ ]:
data.obs['celltype.L1.GBoost'] = df_all_labels['labels'].fillna("Unknown").values.tolist()

In [ ]:
data

In [ ]:
snap.pl.umap(data, color='leiden', marker_size=3, height=500)


In [ ]:
snap.pl.umap(data, color='sample', marker_size=3, height=500)


In [ ]:
data.close()